# CDK2/CDK9/CDK1/CDK6/CDK5/CDK7の高活性化合物と代表的なPDB構造

## Why

`cdk20_similar_targets.ipynb` ranked CDK20's human paralogs by actual data
availability (not just sequence identity) and found six that are, unlike
CDK20 itself (zero PDB structures, ~1 usable ChEMBL activity), rich in both
solved structures and bioactivity data: `CDK2_HUMAN` (522 structures / 3454
activities), `CDK9_HUMAN` (28/2247), `CDK1_HUMAN` (14/1790), `CDK6_HUMAN`
(22/1053), `CDK5_HUMAN` (10/848), `CDK7_HUMAN` (56/824).

This notebook goes one level deeper on those six: (1) pull each target's
most potent ChEMBL compounds, and (2) get an actual 3D PDB structure for
each -- picking a handful of best-resolution entries per target rather than
downloading everything (CDK2 alone has 522), then rendering one
ligand-bound representative per target.

## 1. Most potent compounds from ChEMBL

`chem.chembl.download_activities` with `normalize_smiles=True` (same
pattern as `example_chembl.ipynb`'s BRAF example) desalts/standardizes each
compound via the ChEMBL Structure Pipeline and aggregates duplicate
assay records into one row per unique compound
(`n`/`pchembl_mean`/`pchembl_median`/`pchembl_std`) -- exactly what's needed
to rank compounds by potency rather than by raw assay count. `mw=[250, 650]`
matches the same roughly drug-like range as that example. One tsv per
target, skipped on re-run if already downloaded.

In [ ]:
import os

from chem import chembl

TARGETS = ["CDK2_HUMAN", "CDK9_HUMAN", "CDK1_HUMAN", "CDK6_HUMAN", "CDK5_HUMAN", "CDK7_HUMAN"]
CHEMBL_OUTDIR = "cdk_paralogs_chembl_data"
os.makedirs(CHEMBL_OUTDIR, exist_ok=True)

for target in TARGETS:
    chembl.download_activities(
        target,
        mw=[250, 650],
        normalize_smiles=True,
        output=os.path.join(CHEMBL_OUTDIR, f"{target}.tsv"),
    )

### Top 10 most potent compounds per target

Each target's tsv sorted by `pchembl_mean` descending, top 10 kept, with a
`target` column added so all six can be shown as one table.

In [ ]:
import pandas as pd

TOP_N = 10

top_compounds = []
for target in TARGETS:
    df = pd.read_csv(os.path.join(CHEMBL_OUTDIR, f"{target}.tsv"), sep="\t")
    top = df.sort_values("pchembl_mean", ascending=False).head(TOP_N).copy()
    top.insert(0, "target", target)
    top_compounds.append(top)

top_compounds_df = pd.concat(top_compounds, ignore_index=True)
display(
    top_compounds_df[["target", "parent_chembl_id", "smiles", "n", "pchembl_mean", "mw"]]
    .style.hide(axis="index")
    .format({"pchembl_mean": "{:.2f}", "mw": "{:.2f}"})
)

### Top 5 structures per target, drawn

Same `MolsToGridImage` pattern as `example_chembl.ipynb`, one grid per
target.

In [ ]:
from rdkit import Chem
from rdkit.Chem.Draw import MolsToGridImage

TOP_DRAW_N = 5

for target in TARGETS:
    top = top_compounds_df[top_compounds_df["target"] == target].head(TOP_DRAW_N)
    mols = [Chem.MolFromSmiles(smi) for smi in top["smiles"]]
    legends = [f"{cid} pchembl_mean={v:.2f}" for cid, v in zip(top["parent_chembl_id"], top["pchembl_mean"])]
    print(target)
    display(MolsToGridImage(mols, molsPerRow=5, subImgSize=(220, 180), legends=legends))

## 2. Representative 3D PDB structures

`chem.rcsb.download_structures` has no "top N" option of its own -- it
downloads every matching entry, which is fine for a target with a handful of
structures but wasteful here (CDK2 alone has 522). Since only one
representative structure per target is needed for viewing, its entry-id
search and resolution metadata are used directly, without downloading
anything yet: `chem.rcsb.fetch._search_entry_ids` (RCSB Search API) and
`_fetch_resolutions` (RCSB GraphQL API) -- the same two helpers
`cdk20_similar_targets.ipynb` used for its structure-count cross-check. The
`N_STRUCTURES` best-resolution entries per target are picked this way, then
handed to `download_structures` as an explicit PDB id list (skips target
resolution/search entirely per its own docs) so only those files are
actually fetched.

In [ ]:
from chem.ids import resolve_uniprot_accession
from chem.rcsb.fetch import _fetch_resolutions, _search_entry_ids


def best_resolution_entries(target, n):
    """The n best-resolution (lowest Angstrom) PDB entry ids for target, found via
    entry-id search + resolution metadata only -- no structure files downloaded.
    """
    accession = resolve_uniprot_accession(target)
    entry_ids = _search_entry_ids(accession)
    resolutions = _fetch_resolutions(entry_ids)
    resolved = [eid for eid in entry_ids if resolutions.get(eid) is not None]
    resolved.sort(key=lambda eid: resolutions[eid])
    return resolved[:n]

In [ ]:
from chem import rcsb

PDB_OUTDIR = "cdk_paralogs_pdb_data"
N_STRUCTURES = 3  # best-resolution entries per target, enough to find one ligand-bound representative

chosen_entries = {}
for target in TARGETS:
    entries = best_resolution_entries(target, N_STRUCTURES)
    chosen_entries[target] = entries
    rcsb.download_structures(entries, filetype="pdb", outdir=os.path.join(PDB_OUTDIR, target))

chosen_entries

### Viewing one ligand-bound structure per target

Among each target's `N_STRUCTURES` downloaded candidates, the first with at
least one non-solvent ligand (`chem.ligand.list_ligand_codes`) is shown --
falling back to the first file if none of them happen to be ligand-bound.
`view3d.render_protein`'s caption reports the chosen PDB id, chains, ligand
codes, and resolution for each.

In [ ]:
import glob

from chem import ligand, view3d

for target in TARGETS:
    target_dir = os.path.join(PDB_OUTDIR, target)
    pdb_files = sorted(glob.glob(os.path.join(target_dir, "*.pdb")))
    ligand_bound = [f for f in pdb_files if ligand.list_ligand_codes(f)]
    chosen = ligand_bound[0] if ligand_bound else pdb_files[0]
    print(target)
    view3d.render_protein(chosen)

## Summary

For all six data-rich CDK20 paralogs, this notebook now has, on disk and
ready for downstream SAR/docking work: the top 10 most potent ChEMBL
compounds per target (`cdk_paralogs_chembl_data/`), and 3 best-resolution
PDB structures per target (`cdk_paralogs_pdb_data/`), with one ligand-bound
structure per target rendered above as a concrete starting point for
structure-based follow-up.